We have a restaurant time-series dataset used to predict daily customer demand. It includes historical visitor numbers, restaurant type and location, calendar details such as weekdays and holidays, and reservation information where available. For our project, we are focusing on the top three restaurants with the most historical data to forecast future visitors and support smarter ingredient purchasing to reduce food waste.

Now what we are going to do is first write the code to calculate and find the top 3 restaurants present in this data which has the highest amount of data.

In [13]:
import pandas as pd
from pathlib import Path

# Path to your dataset folder
DATA_DIR = Path("recruit-restaurant-visitor-forecasting")

# Load the main visitor dataset
air_visit_data = pd.read_csv(DATA_DIR / "air_visit_data.csv")

# Count how many daily visitor records each restaurant has
restaurant_record_counts = (
    air_visit_data
    .groupby("air_store_id")
    .size()
    .reset_index(name="number_of_records")
    .sort_values(by="number_of_records", ascending=False)
)

# Get the top 3 restaurants with the most data
top_3_restaurants = restaurant_record_counts.head(3)

top_3_restaurants

,air_store_id,number_of_records
274,air_5c817ef28f236bdf,477
161,air_36bcf77d3382d36e,476
525,air_a083834e7ffe187e,476


Now we want to extract the data of the top 3 restaurants only

In [14]:
# Create output folder
OUTPUT_DIR = DATA_DIR / "top_3_data"
OUTPUT_DIR.mkdir(exist_ok=True)

# Get top 3 restaurant IDs
top_3_ids = top_3_restaurants["air_store_id"].tolist()

print("Top 3 restaurant IDs:")
print(top_3_ids)

# Load only Air-related files
air_store_info = pd.read_csv(DATA_DIR / "air_store_info.csv")
air_reserve = pd.read_csv(DATA_DIR / "air_reserve.csv")
date_info = pd.read_csv(DATA_DIR / "date_info.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

# 1. Copy visitor data for top 3 restaurants
air_visit_top3 = air_visit_data[
    air_visit_data["air_store_id"].isin(top_3_ids)
]

air_visit_top3.to_csv(OUTPUT_DIR / "air_visit_data_top3.csv", index=False)

# 2. Copy store information for top 3 restaurants
air_store_info_top3 = air_store_info[
    air_store_info["air_store_id"].isin(top_3_ids)
]

air_store_info_top3.to_csv(OUTPUT_DIR / "air_store_info_top3.csv", index=False)

# 3. Copy Air reservation data for top 3 restaurants
air_reserve_top3 = air_reserve[
    air_reserve["air_store_id"].isin(top_3_ids)
]

air_reserve_top3.to_csv(OUTPUT_DIR / "air_reserve_top3.csv", index=False)

# 4. Copy sample submission rows for top 3 restaurants
sample_submission_copy = sample_submission.copy()

sample_submission_copy["air_store_id"] = sample_submission_copy["id"].str[:-11]
sample_submission_copy["visit_date"] = sample_submission_copy["id"].str[-10:]

sample_submission_top3 = sample_submission_copy[
    sample_submission_copy["air_store_id"].isin(top_3_ids)
]

sample_submission_top3.to_csv(OUTPUT_DIR / "sample_submission_top3.csv", index=False)

# 5. Copy date info for historical and future dates
historical_dates = set(air_visit_top3["visit_date"].astype(str))
future_dates = set(sample_submission_top3["visit_date"].astype(str))

required_dates = historical_dates.union(future_dates)

date_info_top3 = date_info[
    date_info["calendar_date"].astype(str).isin(required_dates)
]

date_info_top3.to_csv(OUTPUT_DIR / "date_info_top3.csv", index=False)

# 6. Save selected restaurant list
top_3_restaurants.to_csv(OUTPUT_DIR / "top_3_restaurants.csv", index=False)

# 7. Print summary
print("Done. Air-only top 3 data copied into:", OUTPUT_DIR)

print("\nRows copied:")
print("air_visit_data_top3:", len(air_visit_top3))
print("air_store_info_top3:", len(air_store_info_top3))
print("air_reserve_top3:", len(air_reserve_top3))
print("sample_submission_top3:", len(sample_submission_top3))
print("date_info_top3:", len(date_info_top3))

Top 3 restaurant IDs:
['air_5c817ef28f236bdf', 'air_36bcf77d3382d36e', 'air_a083834e7ffe187e']
Done. Air-only top 3 data copied into: recruit-restaurant-visitor-forecasting/top_3_data

Rows copied:
air_visit_data_top3: 1429
air_store_info_top3: 3
air_reserve_top3: 961
sample_submission_top3: 117
date_info_top3: 517


## Now we have to merge the top3 data


### Load Data

In [15]:
import pandas as pd
from pathlib import Path

# Folder where your clean top 3 data is saved
DATA_DIR = Path("top_3_data")

# Load files
air_visit = pd.read_csv(DATA_DIR / "air_visit_data_top3.csv")
air_store = pd.read_csv(DATA_DIR / "air_store_info_top3.csv")
air_reserve = pd.read_csv(DATA_DIR / "air_reserve_top3.csv")
date_info = pd.read_csv(DATA_DIR / "date_info_top3.csv")

print("air_visit:", air_visit.shape)
print("air_store:", air_store.shape)
print("air_reserve:", air_reserve.shape)
print("date_info:", date_info.shape)

air_visit: (1429, 3)
air_store: (3, 5)
air_reserve: (961, 4)
date_info: (517, 3)


### prepare visitor data


In [16]:
# Convert visit_date to datetime
air_visit["visit_date"] = pd.to_datetime(air_visit["visit_date"])

# Sort properly
air_visit = air_visit.sort_values(["air_store_id", "visit_date"])

air_visit.head()

,air_store_id,visit_date,visitors
953,air_36bcf77d3382d36e,2016-01-01,34
954,air_36bcf77d3382d36e,2016-01-02,43
955,air_36bcf77d3382d36e,2016-01-03,30
956,air_36bcf77d3382d36e,2016-01-05,38
957,air_36bcf77d3382d36e,2016-01-06,22


### Prepare date/calender data

In [17]:
# Convert calendar date
date_info["calendar_date"] = pd.to_datetime(date_info["calendar_date"])

# Rename calendar_date to visit_date so we can merge
date_info = date_info.rename(columns={"calendar_date": "visit_date"})

# Create useful calendar features
date_info["month"] = date_info["visit_date"].dt.month
date_info["day"] = date_info["visit_date"].dt.day
date_info["year"] = date_info["visit_date"].dt.year
date_info["week_of_year"] = date_info["visit_date"].dt.isocalendar().week.astype(int)

# Convert day names into numbers
day_map = {
    "Monday": 0,
    "Tuesday": 1,
    "Wednesday": 2,
    "Thursday": 3,
    "Friday": 4,
    "Saturday": 5,
    "Sunday": 6
}

date_info["day_of_week_num"] = date_info["day_of_week"].map(day_map)

# Weekend feature
date_info["is_weekend"] = date_info["day_of_week"].isin(["Saturday", "Sunday"]).astype(int)

date_info.head()

,visit_date,day_of_week,holiday_flg,month,day,year,week_of_year,day_of_week_num,is_weekend
0,2016-01-01,Friday,1,1,1,2016,53,4,0
1,2016-01-02,Saturday,1,1,2,2016,53,5,1
2,2016-01-03,Sunday,1,1,3,2016,53,6,1
3,2016-01-04,Monday,0,1,4,2016,1,0,0
4,2016-01-05,Tuesday,0,1,5,2016,1,1,0


### Prepare Rservation data

In [18]:
# Convert datetime columns
air_reserve["visit_datetime"] = pd.to_datetime(air_reserve["visit_datetime"])
air_reserve["reserve_datetime"] = pd.to_datetime(air_reserve["reserve_datetime"])

# Create visit_date from visit_datetime
air_reserve["visit_date"] = air_reserve["visit_datetime"].dt.date
air_reserve["visit_date"] = pd.to_datetime(air_reserve["visit_date"])

# Calculate how many days before the visit the reservation was made
air_reserve["reserve_lead_days"] = (
    air_reserve["visit_datetime"] - air_reserve["reserve_datetime"]
).dt.total_seconds() / (60 * 60 * 24)

# Aggregate reservation data by restaurant and visit date
reserve_daily = (
    air_reserve
    .groupby(["air_store_id", "visit_date"])
    .agg(
        reserved_visitors_sum=("reserve_visitors", "sum"),
        reservation_count=("reserve_visitors", "count"),
        reserved_visitors_mean=("reserve_visitors", "mean"),
        avg_reserve_lead_days=("reserve_lead_days", "mean")
    )
    .reset_index()
)

reserve_daily.head()

,air_store_id,visit_date,reserved_visitors_sum,reservation_count,reserved_visitors_mean,avg_reserve_lead_days
0,air_5c817ef28f236bdf,2016-01-29,4,1,4.0,1.125000
1,air_5c817ef28f236bdf,2016-01-30,2,1,2.0,0.625000
2,air_5c817ef28f236bdf,2016-02-19,15,1,15.0,1.000000
3,air_5c817ef28f236bdf,2016-02-20,2,1,2.0,0.125000
4,air_5c817ef28f236bdf,2016-02-23,14,1,14.0,0.166667


## Merging everything into one

In [19]:
# Start with main visitor data
model_data = air_visit.copy()

# Add calendar/date features
model_data = model_data.merge(
    date_info,
    on="visit_date",
    how="left"
)

# Add restaurant information
model_data = model_data.merge(
    air_store,
    on="air_store_id",
    how="left"
)

# Add reservation information
model_data = model_data.merge(
    reserve_daily,
    on=["air_store_id", "visit_date"],
    how="left"
)

# Fill missing reservation values with 0
reservation_cols = [
    "reserved_visitors_sum",
    "reservation_count",
    "reserved_visitors_mean",
    "avg_reserve_lead_days"
]

model_data[reservation_cols] = model_data[reservation_cols].fillna(0)

model_data.head()

,air_store_id,visit_date,visitors,day_of_week,holiday_flg,month,day,year,week_of_year,day_of_week_num,is_weekend,air_genre_name,air_area_name,latitude,longitude,reserved_visitors_sum,reservation_count,reserved_visitors_mean,avg_reserve_lead_days
0,air_36bcf77d3382d36e,2016-01-01,34,Friday,1,1,1,2016,53,4,0,Bar/Cocktail,Tōkyō-to Chiyoda-ku Kudanminami,35.694003,139.753595,0.0,0.0,0.0,0.0
1,air_36bcf77d3382d36e,2016-01-02,43,Saturday,1,1,2,2016,53,5,1,Bar/Cocktail,Tōkyō-to Chiyoda-ku Kudanminami,35.694003,139.753595,0.0,0.0,0.0,0.0
2,air_36bcf77d3382d36e,2016-01-03,30,Sunday,1,1,3,2016,53,6,1,Bar/Cocktail,Tōkyō-to Chiyoda-ku Kudanminami,35.694003,139.753595,0.0,0.0,0.0,0.0
3,air_36bcf77d3382d36e,2016-01-05,38,Tuesday,0,1,5,2016,1,1,0,Bar/Cocktail,Tōkyō-to Chiyoda-ku Kudanminami,35.694003,139.753595,0.0,0.0,0.0,0.0
4,air_36bcf77d3382d36e,2016-01-06,22,Wednesday,0,1,6,2016,1,2,0,Bar/Cocktail,Tōkyō-to Chiyoda-ku Kudanminami,35.694003,139.753595,0.0,0.0,0.0,0.0


## Adding feature that shows whether a rstaurat has reservation data

In [20]:
# Find restaurants that have at least one reservation record
restaurants_with_reserve = air_reserve["air_store_id"].unique()

model_data["has_reservation_data"] = model_data["air_store_id"].isin(
    restaurants_with_reserve
).astype(int)

model_data[["air_store_id", "visit_date", "visitors", "reserved_visitors_sum", "has_reservation_data"]].head()

,air_store_id,visit_date,visitors,reserved_visitors_sum,has_reservation_data
0,air_36bcf77d3382d36e,2016-01-01,34,0.0,0
1,air_36bcf77d3382d36e,2016-01-02,43,0.0,0
2,air_36bcf77d3382d36e,2016-01-03,30,0.0,0
3,air_36bcf77d3382d36e,2016-01-05,38,0.0,0
4,air_36bcf77d3382d36e,2016-01-06,22,0.0,0


### Create Previous Visitor Features

In [21]:
# Sort before creating lag features
model_data = model_data.sort_values(["air_store_id", "visit_date"])

# Previous visitor values
model_data["visitors_lag_1"] = model_data.groupby("air_store_id")["visitors"].shift(1)
model_data["visitors_lag_7"] = model_data.groupby("air_store_id")["visitors"].shift(7)
model_data["visitors_lag_14"] = model_data.groupby("air_store_id")["visitors"].shift(14)
model_data["visitors_lag_28"] = model_data.groupby("air_store_id")["visitors"].shift(28)

# Rolling averages
model_data["visitors_rolling_7_mean"] = (
    model_data
    .groupby("air_store_id")["visitors"]
    .transform(lambda x: x.shift(1).rolling(window=7).mean())
)

model_data["visitors_rolling_14_mean"] = (
    model_data
    .groupby("air_store_id")["visitors"]
    .transform(lambda x: x.shift(1).rolling(window=14).mean())
)

model_data["visitors_rolling_28_mean"] = (
    model_data
    .groupby("air_store_id")["visitors"]
    .transform(lambda x: x.shift(1).rolling(window=28).mean())
)

model_data.head()

,air_store_id,visit_date,visitors,day_of_week,holiday_flg,month,day,year,week_of_year,day_of_week_num,...,reserved_visitors_mean,avg_reserve_lead_days,has_reservation_data,visitors_lag_1,visitors_lag_7,visitors_lag_14,visitors_lag_28,visitors_rolling_7_mean,visitors_rolling_14_mean,visitors_rolling_28_mean
0,air_36bcf77d3382d36e,2016-01-01,34,Friday,1,1,1,2016,53,4,...,0.0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,air_36bcf77d3382d36e,2016-01-02,43,Saturday,1,1,2,2016,53,5,...,0.0,0.0,0,34.0,NaN,NaN,NaN,NaN,NaN,NaN
2,air_36bcf77d3382d36e,2016-01-03,30,Sunday,1,1,3,2016,53,6,...,0.0,0.0,0,43.0,NaN,NaN,NaN,NaN,NaN,NaN
3,air_36bcf77d3382d36e,2016-01-05,38,Tuesday,0,1,5,2016,1,1,...,0.0,0.0,0,30.0,NaN,NaN,NaN,NaN,NaN,NaN
4,air_36bcf77d3382d36e,2016-01-06,22,Wednesday,0,1,6,2016,1,2,...,0.0,0.0,0,38.0,NaN,NaN,NaN,NaN,NaN,NaN


## Remove rows with missing lag

In [22]:
model_data_clean = model_data.dropna().reset_index(drop=True)

print("Before removing missing lag rows:", model_data.shape)
print("After removing missing lag rows:", model_data_clean.shape)

model_data_clean.head()

Before removing missing lag rows: (1429, 27)
After removing missing lag rows: (1345, 27)


,air_store_id,visit_date,visitors,day_of_week,holiday_flg,month,day,year,week_of_year,day_of_week_num,...,reserved_visitors_mean,avg_reserve_lead_days,has_reservation_data,visitors_lag_1,visitors_lag_7,visitors_lag_14,visitors_lag_28,visitors_rolling_7_mean,visitors_rolling_14_mean,visitors_rolling_28_mean
0,air_36bcf77d3382d36e,2016-01-30,45,Saturday,0,1,30,2016,4,5,...,0.0,0.0,0,31.0,51.0,50.0,34.0,30.714286,28.357143,28.928571
1,air_36bcf77d3382d36e,2016-01-31,56,Sunday,0,1,31,2016,4,6,...,0.0,0.0,0,45.0,47.0,49.0,43.0,29.857143,28.000000,29.321429
2,air_36bcf77d3382d36e,2016-02-01,17,Monday,0,2,1,2016,5,0,...,0.0,0.0,0,56.0,30.0,11.0,30.0,31.142857,28.500000,29.785714
3,air_36bcf77d3382d36e,2016-02-02,9,Tuesday,0,2,2,2016,5,1,...,0.0,0.0,0,17.0,14.0,20.0,38.0,29.285714,28.928571,29.321429
4,air_36bcf77d3382d36e,2016-02-03,16,Wednesday,0,2,3,2016,5,2,...,0.0,0.0,0,9.0,20.0,17.0,22.0,28.571429,28.142857,28.285714


## Save the Dataset

In [23]:
model_data_clean.to_csv(DATA_DIR / "top3_model_ready_training_data.csv", index=False)

print("Model-ready training data saved successfully.")
print("Saved file:", DATA_DIR / "top3_model_ready_training_data.csv")

Model-ready training data saved successfully.
Saved file: top_3_data/top3_model_ready_training_data.csv
